
# Single LLM-Based AI Agent for Intelligent Job Search and Resume Tailoring

This notebook implements a **single LLM-based AI agent** that performs:

1. **Job filtering**
2. **Job ranking**
3. **Best job selection**
4. **Resume tailoring**

The LLM acts as the **reasoning engine** and decides which tool to invoke based on the current task state.

### Available tools
- **Filtering Tool** – removes unsuitable jobs
- **Ranking Tool** – scores and ranks remaining jobs
- **Resume Tailoring Tool** – tailors the resume for the top-ranked role

The workflow typically results in filtering, ranking, and resume tailoring, but the tool invocation is decided dynamically by the LLM rather than through a rigid hard-coded sequence.


# 1. INSTALL / IMPORTS

In [1]:
!pip install -q groq pandas numpy


In [2]:

import json
import re
import ast
import math
import pandas as pd
import numpy as np
import os
import getpass

from groq import Groq
from IPython.display import display, Markdown

# 2. CONFIGURATION

In [3]:
# Ask user for API key (hidden input)
GROQ_API_KEY = getpass.getpass("Enter your Groq API key: ")

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"

client = Groq(api_key=GROQ_API_KEY)

Enter your Groq API key:  ········


# 3. LOAD DATASET

In [4]:
CSV_PATH = "ml_engineer_jobs_dataset_2026-03-14.csv"   # change if needed

df_jobs = pd.read_csv(CSV_PATH)

print("Dataset loaded successfully.")
print(f"Total jobs in dataset: {len(df_jobs)}")
display(df_jobs.head())

Dataset loaded successfully.
Total jobs in dataset: 24


,Job Title,Company,Location,Remote,Required Skills,Years of Experience,Job Description,URL
0,Machine Learning Engineer,Hive,San Francisco,No,"Python, TensorFlow/Caffe/Torch, deep learning,...",1-2 years,Build and deploy deep learning models end-to-e...,https://jobs.lever.co/hive/fb175ecc-b6ba-4242-...
1,Machine Learning Engineer,Kiddom,San Francisco / New York,No,"Python, SQL, Pandas, scikit-learn, XGBoost, Te...",2+ years,"Build ML systems for search, personalization, ...",https://jobs.lever.co/kiddom/ecfe8b67-8322-411...
2,Machine Learning Engineer (LLMs / AI),ghSMART,United States,Yes,"RAG, LLMs, LangChain, LangGraph, MLOps, data p...",5+ years,Use structured leadership data to build AI age...,https://jobs.lever.co/ghsmartjobs/1dc7e2fc-2ae...
3,Machine Learning Engineer,Healx,Cambridge,No,"Python, knowledge graphs, agentic workflows, M...",2-5 years,"Develop ML approaches, knowledge-graph reasoni...",https://jobs.lever.co/healx/5282b5bb-e95e-4f1c...
4,"Machine Learning Engineer, GenAI Technology",Point72,United States,No,"TensorFlow, PyTorch, Scikit-learn, Python, Jav...",3-7 years,Develop scalable AI/ML architectures and colla...,https://boards.greenhouse.io/point72/jobs/8342...


# 4. CANDIDATE PROFILE

In [5]:
CANDIDATE = {
    "name": "Priya Nair",
    "years_of_experience": 3,
    "location": "Houston, TX",
    "open_to_remote": True,
    "preferred_locations": ["Remote", "Houston, TX", "Austin, TX", "Dallas, TX", "New York, NY", "San Francisco, CA"],
    "excluded_companies": ["Meta", "TikTok"],
    "max_experience_required": 5,
    "skills": [
        "Python",
        "Machine Learning",
        "Deep Learning",
        "NLP",
        "Transformers",
        "PyTorch",
        "TensorFlow",
        "Scikit-learn",
        "Pandas",
        "NumPy",
        "SQL",
        "AWS",
        "MLflow",
        "SageMaker",
        "Data Preprocessing",
        "Model Deployment",
        "A/B Testing",
        "Recommendation Systems",
        "Sentiment Analysis",
        "BERT"
    ]
}

# 5. RESUME TEXT

In [6]:
RESUME = """
Priya Nair
Machine Learning Engineer

Professional Summary:
Machine Learning Engineer with 3 years of experience building and deploying scalable ML systems across recommendation, NLP, and predictive analytics use cases. Strong background in Python, deep learning, model optimization, and cloud-based deployment. Experienced in working with cross-functional teams to translate business problems into ML solutions.

Experience:
- Built a real-time product recommendation engine using collaborative filtering, increasing click-through rate by 22%
- Trained and fine-tuned BERT-based NLP models for sentiment analysis on 10M+ customer reviews
- Deployed models to AWS SageMaker with automated retraining pipelines using MLflow
- Reduced model inference latency by 35% through quantization and ONNX optimization
- Developed classification models using Scikit-learn for customer churn prediction (AUC: 0.91)
- Built data preprocessing pipelines using Pandas, NumPy, and Apache Spark
- Collaborated with product teams to define ML metrics and run A/B tests

Skills:
Python, Machine Learning, Deep Learning, NLP, Transformers, PyTorch, TensorFlow, Scikit-learn, Pandas, NumPy, SQL, AWS, MLflow, SageMaker
"""

# 6. HELPER FUNCTIONS

In [7]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def safe_to_list(value):
    """
    Convert various formats of required skills into a Python list.
    Handles:
    - Python list objects
    - stringified lists
    - comma-separated strings
    """
    if isinstance(value, list):
        return [str(v).strip() for v in value if str(v).strip()]
    
    if pd.isna(value):
        return []

    text = str(value).strip()

    # Try parsing stringified Python list
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass

    # Fall back to comma split
    if "," in text:
        return [v.strip() for v in text.split(",") if v.strip()]
    
    return [text] if text else []

def location_matches(job_location, preferred_locations):
    job_location = normalize_text(job_location).lower()
    preferred = [loc.lower() for loc in preferred_locations]
    return any(loc in job_location or job_location in loc for loc in preferred)

def remote_matches(job_remote_value, job_location, candidate_open_to_remote):
    remote_text = f"{normalize_text(job_remote_value)} {normalize_text(job_location)}".lower()
    if "remote" in remote_text and candidate_open_to_remote:
        return True
    return False

def company_excluded(company, excluded_companies):
    company = normalize_text(company).lower()
    excluded = [c.lower() for c in excluded_companies]
    return company in excluded

def parse_experience_value(exp_value):
    """
    Convert a job experience requirement into a numeric value if possible.
    Works for values like:
    - 3
    - 3.0
    - '3'
    - '3 years'
    - '2-4 years'
    """
    if pd.isna(exp_value):
        return None
    
    if isinstance(exp_value, (int, float)):
        return float(exp_value)

    text = str(exp_value).lower().strip()
    nums = re.findall(r"\d+\.?\d*", text)

    if not nums:
        return None
    
    nums = [float(n) for n in nums]
    return max(nums)

def skill_overlap(candidate_skills, job_skills):
    cset = {s.lower().strip() for s in candidate_skills}
    jset = {s.lower().strip() for s in job_skills}
    if not jset:
        return 0.0, []
    matched = sorted(list(cset.intersection(jset)))
    pct = len(matched) / len(jset)
    return pct, matched

# 7. FILTERING TOOL

In [8]:
def filter_jobs(df, candidate):
    """
    Filtering rules:
    - remove excluded companies
    - remove jobs requiring more than candidate max allowed experience
    - keep if remote and candidate is open to remote
    - otherwise keep if location matches preferred locations
    """
    kept_rows = []
    removal_log = []

    for idx, row in df.iterrows():
        company = normalize_text(row.get("Company", ""))
        location = normalize_text(row.get("Location", ""))
        remote_val = normalize_text(row.get("Remote", ""))
        exp_required = parse_experience_value(row.get("Years of Experience", ""))
        
        reasons = []

        # Excluded company
        if company_excluded(company, candidate["excluded_companies"]):
            reasons.append("Excluded company")

        # Experience limit
        if exp_required is not None and exp_required > candidate["max_experience_required"]:
            reasons.append(f"Requires {exp_required} years experience (> {candidate['max_experience_required']})")

        # Location / remote logic
        loc_ok = location_matches(location, candidate["preferred_locations"])
        remote_ok = remote_matches(remote_val, location, candidate["open_to_remote"])

        if not (loc_ok or remote_ok):
            reasons.append("Does not match preferred locations or remote preference")

        if reasons:
            removal_log.append({
                "job_index": int(idx),
                "job_title": normalize_text(row.get("Job Title", "")),
                "company": company,
                "reason": "; ".join(reasons)
            })
        else:
            kept_rows.append(row)

    filtered_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    return filtered_df, removal_log

# 8. RANKING TOOL

In [9]:
# Scoring weights — change these to adjust ranking priorities
SCORE_WEIGHTS = {
    "skill":    60,   # max points for skill match
    "exp":      30,   # max points for experience alignment
    "loc":      10,   # max points for location/remote match
    "loc_pref":  8,   # partial credit for preferred city (non-remote)
}

def rank_jobs(df, candidate, weights=None):
    """
    Score out of 100:
    - Skill Match Score:      0-{skill}
    - Experience Alignment:   0-{exp}
    - Location Match Bonus:   0-{loc}
    Weights are configurable via the SCORE_WEIGHTS dict.
    """.format(**SCORE_WEIGHTS)
    if weights is None:
        weights = SCORE_WEIGHTS

    ranked_rows = []

    for _, row in df.iterrows():
        job_skills = safe_to_list(row.get("Required Skills", ""))
        exp_required = parse_experience_value(row.get("Years of Experience", ""))
        location = normalize_text(row.get("Location", ""))
        remote_val = normalize_text(row.get("Remote", ""))

        # Skill score — scales with weight
        skill_pct, matched_skills = skill_overlap(candidate["skills"], job_skills)
        skill_score = round(skill_pct * weights["skill"], 2)

        # Experience score — scales with weight
        candidate_exp = candidate["years_of_experience"]
        w = weights["exp"]
        if exp_required is None:
            exp_score = round(w * 0.67, 1)          # neutral: ~2/3 of max
        else:
            diff = abs(candidate_exp - exp_required)
            if candidate_exp >= exp_required:
                exp_score = float(w)                 # exact or overqualified
            elif diff <= 1:
                exp_score = round(w * 0.80, 1)       # 1 yr under
            elif diff <= 2:
                exp_score = round(w * 0.60, 1)       # 2 yrs under
            else:
                exp_score = round(w * 0.33, 1)       # too far under

        # Location score
        loc_score = 0.0
        if remote_matches(remote_val, location, candidate["open_to_remote"]):
            loc_score = float(weights["loc"])
        elif location_matches(location, candidate["preferred_locations"]):
            loc_score = float(weights["loc_pref"])

        total_score = round(skill_score + exp_score + loc_score, 2)

        row_copy = row.copy()
        row_copy["Total Score"] = total_score
        row_copy["Score Details"] = {
            "Skill Score": skill_score,
            "Exp Score": exp_score,
            "Loc Score": loc_score,
            "Skill Match %": round(skill_pct * 100, 2),
            "Matched Skills": matched_skills
        }
        ranked_rows.append(row_copy)

    ranked_df = pd.DataFrame(ranked_rows)
    ranked_df = ranked_df.sort_values(by="Total Score", ascending=False).reset_index(drop=True)
    ranked_df["Rank"] = range(1, len(ranked_df) + 1)

    # Reorder columns a bit
    cols = ["Rank"] + [c for c in ranked_df.columns if c != "Rank"]
    ranked_df = ranked_df[cols]
    return ranked_df

# 9. SHARED STATE

In [10]:
state = {
    "df_jobs": df_jobs,
    "df_filtered": None,
    "df_ranked": None,
    "removal_log": None,
    "top_job": None,
    "tailored_resume": None,
    "tailored_raw": None,
    "original_bullets": None,
}

# 10. TOOL SCHEMAS

In [11]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "filter_jobs",
            "description": "Filter unsuitable jobs using candidate constraints such as company exclusion, experience limit, remote preference, and preferred locations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {
                        "type": "string",
                        "description": "Why filtering is needed now."
                    }
                },
                "required": ["reason"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rank_jobs",
            "description": "Score and rank the filtered jobs using skill match, experience alignment, and location match.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {
                        "type": "string",
                        "description": "Why ranking is needed now."
                    }
                },
                "required": ["reason"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tailor_resume",
            "description": "Tailor the resume for the highest-ranked job by rewriting the professional summary and modifying exactly two experience bullet points.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {
                        "type": "string",
                        "description": "Why resume tailoring is needed now."
                    }
                },
                "required": ["reason"]
            }
        }
    }
]

# 11. MESSAGE SERIALIZATION

In [12]:
def serialize_message(msg):
    """
    Convert Groq ChatCompletionMessage object to plain dict.
    """
    if isinstance(msg, dict):
        return msg

    d = {
        "role": msg.role,
        "content": msg.content or ""
    }

    if getattr(msg, "tool_calls", None):
        d["tool_calls"] = [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            }
            for tc in msg.tool_calls
        ]

    return d

def safe_extract_response_text(response):
    try:
        return response.choices[0].message.content or ""
    except Exception:
        return ""

# 12. LLM CALL

In [13]:
def call_llm(messages, tools=None, tool_choice="auto"):
    clean_messages = [serialize_message(m) for m in messages]

    kwargs = {
        "model": MODEL,
        "messages": clean_messages,
        "temperature": 0.3
    }

    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice

    response = client.chat.completions.create(**kwargs)

    reasoning_text = safe_extract_response_text(response)
    # if reasoning_text:
    #     print("\nAgent reasoning:")
    #     print(reasoning_text)

    return response.choices[0].message

# 13. TOOL DISPATCHER

In [14]:
def dispatch_tool(tool_name, args):
    if tool_name == "filter_jobs":
        print("\n--- Invoking Filtering Tool ---")
        filtered_df, removal_log = filter_jobs(state["df_jobs"], CANDIDATE)
        state["df_filtered"] = filtered_df
        state["removal_log"] = removal_log

        result = {
            "jobs_before_filter": len(state["df_jobs"]),
            "jobs_after_filter": len(filtered_df),
            "removed_count": len(removal_log),
            "removal_summary": removal_log,
            "filtered_jobs": [
                {
                    "id": int(i),
                    "title": r["Job Title"],
                    "company": r["Company"],
                    "location": r["Location"],
                    "remote": r["Remote"] if "Remote" in r else "",
                    "experience": r["Years of Experience"],
                    "skills": str(r["Required Skills"])[:120]
                }
                for i, r in filtered_df.iterrows()
            ]
        }

        print(f"Filtered jobs: {result['jobs_before_filter']} → {result['jobs_after_filter']}")
        return result

    elif tool_name == "rank_jobs":
        print("\n--- Invoking Ranking Tool ---")

        if state["df_filtered"] is None:
            return {"error": "Cannot rank before filtering."}

        ranked_df = rank_jobs(state["df_filtered"], CANDIDATE)
        state["df_ranked"] = ranked_df
        state["top_job"] = ranked_df.iloc[0]

        result = {
            "ranked_jobs": [
                {
                    "rank": int(r["Rank"]),
                    "title": r["Job Title"],
                    "company": r["Company"],
                    "location": r["Location"],
                    "score": float(r["Total Score"]),
                    "skill_score": float(r["Score Details"]["Skill Score"]),
                    "exp_score": float(r["Score Details"]["Exp Score"]),
                    "loc_score": float(r["Score Details"]["Loc Score"]),
                    "skill_match": float(r["Score Details"]["Skill Match %"])
                }
                for _, r in ranked_df.iterrows()
            ],
            "top_3": [
                {
                    "rank": int(r["Rank"]),
                    "title": r["Job Title"],
                    "company": r["Company"],
                    "score": float(r["Total Score"])
                }
                for _, r in ranked_df.head(3).iterrows()
            ]
        }

        print("Top 3 ranked jobs:")
        for item in result["top_3"]:
            print(f"Rank {item['rank']}: {item['title']} @ {item['company']} | Score = {item['score']}")

        return result

    elif tool_name == "tailor_resume":
        print("\n--- Invoking Resume Tailoring Tool ---")

        if state["top_job"] is None:
            return {"error": "Cannot tailor resume before identifying the top-ranked job."}

        tj = state["top_job"]

        # Parse bullets dynamically from RESUME — no static list needed
        bullet_lines = [
            line.strip().lstrip("- ").strip()
            for line in RESUME.splitlines()
            if line.strip().startswith("-") and len(line.strip()) > 5
        ]
        state["original_bullets"] = bullet_lines

        numbered_bullets = "\n".join(
            f"{i+1}. {b}" for i, b in enumerate(bullet_lines)
        )
        candidate_name = CANDIDATE["name"]

        tailor_prompt = f"""
You are an expert resume writer. Tailor {candidate_name}'s resume for this specific job.

TOP JOB:
Title       : {tj['Job Title']}
Company     : {tj['Company']}
Location    : {tj['Location']}
Skills Req  : {tj['Required Skills']}
Description : {tj['Job Description']}

CURRENT RESUME:
{RESUME}

ORIGINAL EXPERIENCE BULLETS (numbered for reference):
{numbered_bullets}

INSTRUCTIONS — follow EXACTLY:
1. Rewrite the Professional Summary in 3-4 sentences tailored to this job.
2. Choose EXACTLY 2 bullets from the numbered list above that are most relevant to the role.
3. Rewrite those 2 bullets to better align with the job description.
4. List the top 5 aligned skills from the candidate profile that match this job.
5. Do NOT rewrite the full resume.

Return ONLY valid JSON using this exact structure:
{{
  "tailored_summary": "<rewritten 3-4 sentence summary>",
  "bullet_changes": [
    {{
      "bullet_number": <1-7>,
      "original": "<exact original bullet text>",
      "modified": "<rewritten bullet>",
      "reason": "<why this bullet was chosen>"
    }},
    {{
      "bullet_number": <1-7>,
      "original": "<exact original bullet text>",
      "modified": "<rewritten bullet>",
      "reason": "<why this bullet was chosen>"
    }}
  ],
  "aligned_skills": ["skill1", "skill2", "skill3", "skill4", "skill5"]
}}
"""

        tailor_response = call_llm([{"role": "user", "content": tailor_prompt}])

        raw = (tailor_response.content or "").strip()
        raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
        raw = re.sub(r"```\s*$", "", raw, flags=re.MULTILINE).strip()

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            parsed = {
                "tailored_summary": raw,
                "bullet_changes": [],
                "aligned_skills": [],
                "raw_output": raw
            }

        state["tailored_resume"] = parsed
        state["tailored_raw"] = raw

        print(f"Resume tailored for: {tj['Job Title']} @ {tj['Company']}")
        return {"tailored_resume": parsed}

    else:
        return {"error": f"Unknown tool: {tool_name}"}

# 14. SYSTEM / USER PROMPTS

In [15]:
SYSTEM_PROMPT = f"""
You are an intelligent job search agent.

Your goal is to help the candidate identify the best ML Engineer opportunities from a dataset of job postings and tailor the resume for the strongest match.

You have access to three tools:
1. filter_jobs   — remove unsuitable jobs using candidate constraints
2. rank_jobs     — score and rank relevant jobs
3. tailor_resume — tailor the resume for the best selected job

Your job is to reason about the current state of the task and decide which tool to invoke next.

Guidelines:
- Use tools only when necessary.
- Briefly explain why you are calling a tool before making the tool call.
- After receiving a tool result, explain what you learned and decide the next best action.
- Do not call a tool redundantly unless there is a clear reason.
- Once ranking is complete, summarize the top 3 jobs clearly.
- Tailor the resume only for the best overall job.

Current state logic:
- If jobs have not yet been filtered, consider whether filtering is needed first.
- If filtered jobs exist but no ranking has been done, consider ranking next.
- If a top-ranked job exists, consider whether resume tailoring should be done.
- If enough information is available, stop calling tools and produce the final answer.

CANDIDATE PROFILE:
Name            : {CANDIDATE['name']}
Experience      : {CANDIDATE['years_of_experience']} years
Location        : {CANDIDATE['location']}
Open to Remote  : {CANDIDATE['open_to_remote']}
Skills          : {', '.join(CANDIDATE['skills'])}
Preferred Locs  : {', '.join(CANDIDATE['preferred_locations'])}
Excluded Cos    : {', '.join(CANDIDATE['excluded_companies'])}
Max Exp Required: {CANDIDATE['max_experience_required']} years
"""

USER_MESSAGE = """
I have uploaded a dataset of 24 ML Engineer job postings along with the candidate profile.

Please analyze the jobs, identify the best opportunities for this candidate, and tailor the resume for the strongest match.

Use the available tools as needed.
Show your reasoning at each step.
"""

# 15. AGENT EXECUTION

In [16]:
print("=" * 70)
print(" AGENT STARTING — LLaMA 3.3-70B via Groq")
print("=" * 70)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_MESSAGE},
]

step = 0
MAX_STEPS = 10

while step < MAX_STEPS:
    step += 1
    print(f"\n{'─'*60}")
    print(f" AGENT STEP {step}")
    print(f"{'─'*60}")

    response_msg = call_llm(messages, tools=TOOLS)

    if response_msg.content:
        print(f"\n AGENT REASONING:\n{response_msg.content}")

    if not response_msg.tool_calls:
        print("\n Agent completed all tasks.")
        messages.append({"role": "assistant", "content": response_msg.content or ""})
        break

    messages.append(serialize_message(response_msg))

    for tool_call in response_msg.tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        print(f"\n TOOL CALLED: {tool_name}")
        print(f"Reason: {tool_args.get('reason', 'N/A')}")

        tool_result = dispatch_tool(tool_name, tool_args)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(tool_result)
        })

print("\n" + "=" * 70)
print(" AGENT LOOP COMPLETE")
print("=" * 70)

 AGENT STARTING — LLaMA 3.3-70B via Groq

────────────────────────────────────────────────────────────
 AGENT STEP 1
────────────────────────────────────────────────────────────

 AGENT REASONING:
To begin, I need to filter the job postings based on the candidate's constraints to remove any unsuitable jobs. This is necessary because the candidate has specific preferences, such as location and excluded companies, that need to be considered. 



 TOOL CALLED: filter_jobs
Reason: Initial filtering based on candidate constraints

--- Invoking Filtering Tool ---
Filtered jobs: 24 → 8

────────────────────────────────────────────────────────────
 AGENT STEP 2
────────────────────────────────────────────────────────────

 AGENT REASONING:
After filtering the jobs based on the candidate's constraints, 16 jobs were removed, and 8 jobs remain. The removed jobs did not match the candidate's preferred locations or remote preference, or required more experience than the candidate has.

Next, I will

# 16. OUTPUTS REQUIRED BY ASSIGNMENT

In [17]:
print("\n===== FILTERED JOBS =====")
if state["df_filtered"] is not None:
    display(state["df_filtered"][["Job Title", "Company", "Location", "Years of Experience"]])
else:
    print("No filtered jobs available.")

print("\n===== RANKED JOB LIST WITH SCORES =====")
if state["df_ranked"] is not None:
    display(state["df_ranked"][["Rank", "Job Title", "Company", "Location", "Total Score"]])
else:
    print("No ranked jobs available.")

print("\n===== TOP 3 JOBS =====")
if state["df_ranked"] is not None:
    top3 = state["df_ranked"].head(3)[["Rank", "Job Title", "Company", "Location", "Total Score"]]
    display(top3)
else:
    print("Top 3 jobs not available.")

print("\n===== FINAL AGENT OUTPUT =====")
if state["top_job"] is not None:
    print("Top Job Recommendation :", state["top_job"]["Job Title"])
    print("Company                :", state["top_job"]["Company"])
    print("Location               :", state["top_job"]["Location"])
    print("Score                  :", state["top_job"]["Total Score"])
else:
    print("No top-ranked job selected.")


===== FILTERED JOBS =====


,Job Title,Company,Location,Years of Experience
0,Machine Learning Engineer,Hive,San Francisco,1-2 years
1,Senior Machine Learning Engineer,Reddit,Remote - United States,3-5+ years
2,Machine Learning Engineer (Model Dev),Artera,Remote-US,2+ years
3,Machine Learning Engineer,Interwell Health,"Remote, United States",3+ years ML; 3+ years MLOps
4,Machine Learning Engineer (Contractor),Tech Holding,"USA, Remote",5+ years
5,Machine Learning Engineer,Orchard Robotics,San Francisco,2+ years
6,AI / ML Engineer,Floe Labs,Remote / Everywhere,3 years
7,Machine Learning Engineer,Meltwater,"New York, NY",2+ years



===== RANKED JOB LIST WITH SCORES =====


,Rank,Job Title,Company,Location,Total Score
0,1,Machine Learning Engineer,Orchard Robotics,San Francisco,71.33
1,2,Machine Learning Engineer,Hive,San Francisco,55.14
2,3,Machine Learning Engineer,Interwell Health,"Remote, United States",55.00
3,4,Machine Learning Engineer (Contractor),Tech Holding,"USA, Remote",53.71
4,5,AI / ML Engineer,Floe Labs,Remote / Everywhere,50.00
5,6,Senior Machine Learning Engineer,Reddit,Remote - United States,44.36
6,7,Machine Learning Engineer (Model Dev),Artera,Remote-US,40.00
7,8,Machine Learning Engineer,Meltwater,"New York, NY",38.00



===== TOP 3 JOBS =====


,Rank,Job Title,Company,Location,Total Score
0,1,Machine Learning Engineer,Orchard Robotics,San Francisco,71.33
1,2,Machine Learning Engineer,Hive,San Francisco,55.14
2,3,Machine Learning Engineer,Interwell Health,"Remote, United States",55.00



===== FINAL AGENT OUTPUT =====
Top Job Recommendation : Machine Learning Engineer
Company                : Orchard Robotics
Location               : San Francisco
Score                  : 71.33


# 17. TAILORED RESUME OUTPUT

In [18]:
print("\n===== TAILORED RESUME OUTPUT =====")
if state["tailored_resume"] is not None:
    tr = state["tailored_resume"]

    print("\nTailored Professional Summary:\n")
    print(tr.get("tailored_summary", "N/A"))

    print("\nModified Experience Bullet Points:\n")
    bullet_changes = tr.get("bullet_changes", [])
    if bullet_changes:
        for idx, change in enumerate(bullet_changes, start=1):
            print(f"Change {idx}:")
            print("Original :", change.get("original", "N/A"))
            print("Modified :", change.get("modified", "N/A"))
            print("Reason   :", change.get("reason", "N/A"))
            print("-" * 50)
    else:
        print("No bullet changes found.")

    print("\nAligned Skills:\n")
    print(tr.get("aligned_skills", []))
else:
    print("No tailored resume output available.")


===== TAILORED RESUME OUTPUT =====

Tailored Professional Summary:

Machine Learning Engineer with 3 years of experience building and deploying scalable ML systems, now looking to leverage skills in Python, PyTorch, and MLflow to develop production-grade ML infrastructure for agricultural AI systems. Strong background in deep learning, model optimization, and cloud-based deployment. Experienced in working with cross-functional teams to translate business problems into ML solutions, with a focus on analyzing large-scale data. Proficient in designing and implementing data pipelines and ML systems that can handle complex data sets.

Modified Experience Bullet Points:

Change 1:
Original : Deployed models to AWS SageMaker with automated retraining pipelines using MLflow
Modified : Designed and deployed scalable ML models to cloud platforms using MLflow and AWS SageMaker, ensuring seamless integration with data pipelines and automated retraining for optimal performance, which can be applie

# 18. REASONING TRACE SUMMARY

In [19]:
print("\n===== REASONING TRACE SUMMARY =====")
print("The agent used LLM reasoning at each step to decide which tool to invoke.\n")

step_num = 0
for msg in messages:
    role = msg.get("role", "")

    if role == "assistant":
        content = msg.get("content", "")
        tool_calls = msg.get("tool_calls", [])
        
        if content or tool_calls:
            step_num += 1
            print(f"--- Step {step_num} ---")
        
        if content:
            print(f"Agent reasoning: {content.strip()}")
        
        for tc in tool_calls:
            fn = tc.get("function", {})
            name = fn.get("name", "")
            args = json.loads(fn.get("arguments", "{}"))
            reason = args.get("reason", "")
            print(f"Tool called   : {name}")
            print(f"Reason        : {reason}")
        
        if content or tool_calls:
            print()

    elif role == "tool":
        result_raw = msg.get("content", "{}")
        try:
            result = json.loads(result_raw)
            # Print a short summary, not the full payload
            if "jobs_after_filter" in result:
                print(f"Tool result   : {result['jobs_before_filter']} jobs → {result['jobs_after_filter']} after filter\n")
            elif "top_3" in result:
                top = result["top_3"]
                print(f"Tool result   : Top job = {top[0]['title']} @ {top[0]['company']} (score {top[0]['score']})\n")
            elif "tailored_resume" in result:
                print(f"Tool result   : Resume tailored successfully\n")
        except:
            pass


===== REASONING TRACE SUMMARY =====
The agent used LLM reasoning at each step to decide which tool to invoke.

--- Step 1 ---
Agent reasoning: To begin, I need to filter the job postings based on the candidate's constraints to remove any unsuitable jobs. This is necessary because the candidate has specific preferences, such as location and excluded companies, that need to be considered.
Tool called   : filter_jobs
Reason        : Initial filtering based on candidate constraints

Tool result   : 24 jobs → 8 after filter

--- Step 2 ---
Agent reasoning: After filtering the jobs based on the candidate's constraints, 16 jobs were removed, and 8 jobs remain. The removed jobs did not match the candidate's preferred locations or remote preference, or required more experience than the candidate has.

Next, I will rank the remaining 8 jobs based on their relevance to the candidate's skills and experience.
Tool called   : rank_jobs
Reason        : Ranking the filtered jobs based on skill match,